In [1]:
import pandas as pd
import numpy as np
import os
import time
import glob

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 1000)

In [91]:
directory = '/users/blaizelahman/Desktop/CFB Model/Updated Data'
pattern = os.path.join(directory, '*updated_model*.csv')

teamFiles = glob.glob(pattern)

teamDict = {}

for file in teamFiles:

    teamDF = pd.read_csv(file)

    if teamDF.shape[0] >= 56:
    
        key = teamDF['School'][0]
        teamDict[key] = teamDF
    
        print(f'Added: {key}')

Added: Western Michigan
Added: Memphis
Added: Arkansas
Added: Wake Forest
Added: Texas State
Added: Temple
Added: Auburn
Added: Miami
Added: Alabama
Added: Louisville
Added: Washington State
Added: North Texas
Added: Washington
Added: Virginia Tech
Added: Texas A&M
Added: Buffalo
Added: Ohio
Added: Minnesota
Added: Syracuse
Added: Southern Mississippi
Added: Ole Miss
Added: Idaho
Added: Western Kentucky
Added: Utah
Added: Stanford
Added: Georgia State
Added: Utah State
Added: West Virginia
Added: Nevada
Added: Louisiana
Added: Mississippi State
Added: Boston College
Added: Michigan State
Added: Miami (OH)
Added: Georgia Southern
Added: UT San Antonio
Added: Pittsburgh
Added: Nebraska
Added: Clemson
Added: Troy
Added: Arizona State
Added: Kent State
Added: UTEP
Added: Toledo
Added: UCLA
Added: Louisiana Monroe
Added: Oregon
Added: Oklahoma State
Added: Kansas State
Added: Middle Tennessee
Added: South Florida
Added: USC
Added: South Alabama
Added: Hawai'i
Added: Bowling Green
Added: Pen

In [3]:
# setting the download directory and Chrome settings
directory = "/Users/blaizelahman/Desktop/CFBData"
chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory": directory}
chromeOptions.add_experimental_option("prefs", prefs)

# creating Chrome driver
driver = webdriver.Chrome(service = Service(ChromeDriverManager().install()), options = chromeOptions)

# making a dictionary to hold team betting data
bettingDict = {}

# downloading betting data for years 2035-2023 from collegefootballdata.com
for year in range(2013, 2024):

    # skipping 2020 because it has bad data
    if year == 2020: 
        continue
        
    try:
        url = f'https://collegefootballdata.com/exporter/lines?year={year}&seasonType=regular'
        driver.get(url)
        time.sleep(4) 
            
        # clicking the query button
        query = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Query')]")
        query.click()
        time.sleep(3) 
            
        # clicking the export button
        export = driver.find_element(By.XPATH, "//button[contains(span/text(), 'Export')]")
        export.click()
        time.sleep(3)

        key = str(year)

            
        # grabs files from CFBData folder
        files = os.listdir(directory)
        
        # grab the file paths for all files ending in .csv
        filePaths = [os.path.join(directory, name) for name in files if name.endswith('.csv')]

        # grabbing the most recently made file out of those in paths
        file = max(filePaths, key=os.path.getctime)
            
        # loading csv file
        bettingDict[key] = pd.read_csv(file)

        # deleting the file after it has been added
        os.remove(file)

    except Exception as e:
        print(f'Cannot grab data for {year}. Error: {e}')

    print(f'Successfully grabbed data for {year}')

driver.quit()

Successfully grabbed data for 2013
Successfully grabbed data for 2014
Successfully grabbed data for 2015
Successfully grabbed data for 2016
Successfully grabbed data for 2017
Successfully grabbed data for 2018
Successfully grabbed data for 2019
Successfully grabbed data for 2021
Successfully grabbed data for 2022
Successfully grabbed data for 2023


In [92]:
# setting our preferred line providers
lineProviders = ['DraftKings', 'consensus', 'Bovada']

# combining betting dataframes
combinedBetting = pd.concat(bettingDict.values(), ignore_index = True)

# function to get line from most preferred available line
def getPreferredLine(group):

    # go through preferred lines and if our preferred providers are there, return the line
    for provider in lineProviders:
        preferredLine = group[group['LineProvider'] == provider]

        # if preferred line provider has line, return
        if not preferredLine.empty:
            return preferredLine.iloc[0]

    # returning first available line if none of our preferred ones are available
    return group.iloc[0] 

# getting the preferred line for each individual game
preferredLines = combinedBetting.groupby('Id').apply(lambda x: getPreferredLine(x)).reset_index(drop = True)

# merging the preferred lines with each dataframe in teamDict
for team, teamDF in teamDict.items():
    
    # merging based on Game Id
    teamDict[team] = teamDF.merge(preferredLines, left_on = 'Game Id', right_on = 'Id', how = 'left')

In [93]:
teamDict['Florida State'][teamDict['Florida State']['Year'] == 2023]

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Game Id,School,Conference,HomeAway,Points,Week,Year,completionAttempts,defensiveTDs,firstDowns,fourthDownEff,fumblesLost,fumblesRecovered,interceptionTDs,interceptionYards,interceptions,kickReturnTDs,kickReturnYards,kickReturns,kickingPoints,netPassingYards,passesDeflected,passesIntercepted,passingTDs,possessionTime,puntReturnTDs,puntReturnYards,puntReturns,qbHurries,rushingAttempts,rushingTDs,rushingYards,sacks,tackles,tacklesForLoss,thirdDownEff,totalFumbles,totalPenaltiesYards,totalYards,turnovers,yardsPerPass,yardsPerRushAttempt,totalTDs,School_opp,Conference_opp,HomeAway_opp,Points_opp,Week_opp,Year_opp,completionAttempts_opp,defensiveTDs_opp,firstDowns_opp,fourthDownEff_opp,fumblesLost_opp,fumblesRecovered_opp,interceptionTDs_opp,interceptionYards_opp,interceptions_opp,kickReturnTDs_opp,kickReturnYards_opp,kickReturns_opp,kickingPoints_opp,netPassingYards_opp,passesDeflected_opp,passesIntercepted_opp,passingTDs_opp,possessionTime_opp,puntReturnTDs_opp,puntReturnYards_opp,puntReturns_opp,qbHurries_opp,rushingAttempts_opp,rushingTDs_opp,rushingYards_opp,sacks_opp,tackles_opp,tacklesForLoss_opp,thirdDownEff_opp,totalFumbles_opp,totalPenaltiesYards_opp,totalYards_opp,turnovers_opp,yardsPerPass_opp,yardsPerRushAttempt_opp,totalTDs_opp,scoreDiff,pointTotal,Win,rolling_sum_Points20,rolling_sum_firstDowns20,rolling_sum_fumblesLost20,rolling_sum_fumblesRecovered20,rolling_sum_interceptions20,rolling_sum_kickReturnYards20,rolling_sum_kickingPoints20,rolling_sum_netPassingYards20,rolling_sum_passesDeflected20,rolling_sum_passesIntercepted20,rolling_sum_passingTDs20,rolling_sum_puntReturns20,rolling_sum_qbHurries20,rolling_sum_rushingAttempts20,rolling_sum_rushingTDs20,rolling_sum_rushingYards20,rolling_sum_sacks20,rolling_sum_tacklesForLoss20,rolling_sum_totalFumbles20,rolling_sum_totalPenaltiesYards20,rolling_sum_totalYards20,rolling_sum_turnovers20,rolling_sum_yardsPerPass20,rolling_sum_yardsPerRushAttempt20,rolling_sum_totalTDs20,rolling_sum_Points8,rolling_sum_firstDowns8,rolling_sum_fumblesLost8,rolling_sum_fumblesRecovered8,rolling_sum_interceptions8,rolling_sum_kickReturnYards8,rolling_sum_kickingPoints8,rolling_sum_netPassingYards8,rolling_sum_passesDeflected8,rolling_sum_passesIntercepted8,rolling_sum_passingTDs8,rolling_sum_puntReturns8,rolling_sum_qbHurries8,rolling_sum_rushingAttempts8,rolling_sum_rushingTDs8,rolling_sum_rushingYards8,rolling_sum_sacks8,rolling_sum_tacklesForLoss8,rolling_sum_totalFumbles8,rolling_sum_totalPenaltiesYards8,rolling_sum_totalYards8,rolling_sum_turnovers8,rolling_sum_yardsPerPass8,rolling_sum_yardsPerRushAttempt8,rolling_sum_totalTDs8,rolling_sum_Points20_opp,rolling_sum_firstDowns20_opp,rolling_sum_fumblesLost20_opp,rolling_sum_fumblesRecovered20_opp,rolling_sum_interceptions20_opp,rolling_sum_kickReturnYards20_opp,rolling_sum_kickingPoints20_opp,rolling_sum_netPassingYards20_opp,rolling_sum_passesDeflected20_opp,rolling_sum_passesIntercepted20_opp,rolling_sum_passingTDs20_opp,rolling_sum_puntReturns20_opp,rolling_sum_qbHurries20_opp,rolling_sum_rushingAttempts20_opp,rolling_sum_rushingTDs20_opp,rolling_sum_rushingYards20_opp,rolling_sum_sacks20_opp,rolling_sum_tacklesForLoss20_opp,rolling_sum_totalFumbles20_opp,rolling_sum_totalPenaltiesYards20_opp,rolling_sum_totalYards20_opp,rolling_sum_turnovers20_opp,rolling_sum_yardsPerPass20_opp,rolling_sum_yardsPerRushAttempt20_opp,rolling_sum_totalTDs20_opp,rolling_sum_Points8_opp,rolling_sum_firstDowns8_opp,rolling_sum_fumblesLost8_opp,rolling_sum_fumblesRecovered8_opp,rolling_sum_interceptions8_opp,rolling_sum_kickReturnYards8_opp,rolling_sum_kickingPoints8_opp,rolling_sum_netPassingYards8_opp,rolling_sum_passesDeflected8_opp,rolling_sum_passesIntercepted8_opp,rolling_sum_passingTDs8_opp,rolling_sum_puntReturns8_opp,rolling_sum_qbHurries8_opp,rolling_sum_rushingAttempts8_opp,rolling_sum_rushingTDs8_opp,rolling_sum_rushingYards8_opp,rolling_sum_sacks8_opp,rolling_sum_tacklesForLoss8_opp,rolling_sum_totalFumbl

In [94]:
def correctSpread(row, team):
    if row['HomeTeam'] == team:
        row['Spread'] = float(row['Spread']) * -1
    return row

In [95]:
for team, teamDF in teamDict.items(): 
    
    teamDict[team] = teamDF.apply(correctSpread, axis = 1, team = team)

In [101]:
teamDict['TCU'][teamDict['TCU']['Year'] == 2023]

,Game Id,School,Conference,HomeAway,Points,Week,Year,completionAttempts,defensiveTDs,firstDowns,fourthDownEff,fumblesLost,fumblesRecovered,interceptionTDs,interceptionYards,interceptions,kickReturnTDs,kickReturnYards,kickReturns,kickingPoints,netPassingYards,passesDeflected,passesIntercepted,passingTDs,possessionTime,puntReturnTDs,puntReturnYards,puntReturns,qbHurries,rushingAttempts,rushingTDs,rushingYards,sacks,tackles,tacklesForLoss,thirdDownEff,totalFumbles,totalPenaltiesYards,totalYards,turnovers,yardsPerPass,yardsPerRushAttempt,totalTDs,School_opp,Conference_opp,HomeAway_opp,Points_opp,Week_opp,Year_opp,completionAttempts_opp,defensiveTDs_opp,firstDowns_opp,fourthDownEff_opp,fumblesLost_opp,fumblesRecovered_opp,interceptionTDs_opp,interceptionYards_opp,interceptions_opp,kickReturnTDs_opp,kickReturnYards_opp,kickReturns_opp,kickingPoints_opp,netPassingYards_opp,passesDeflected_opp,passesIntercepted_opp,passingTDs_opp,possessionTime_opp,puntReturnTDs_opp,puntReturnYards_opp,puntReturns_opp,qbHurries_opp,rushingAttempts_opp,rushingTDs_opp,rushingYards_opp,sacks_opp,tackles_opp,tacklesForLoss_opp,thirdDownEff_opp,totalFumbles_opp,totalPenaltiesYards_opp,totalYards_opp,turnovers_opp,yardsPerPass_opp,yardsPerRushAttempt_opp,totalTDs_opp,scoreDiff,pointTotal,Win,rolling_sum_Points20,rolling_sum_firstDowns20,rolling_sum_fumblesLost20,rolling_sum_fumblesRecovered20,rolling_sum_interceptions20,rolling_sum_kickReturnYards20,rolling_sum_kickingPoints20,rolling_sum_netPassingYards20,rolling_sum_passesDeflected20,rolling_sum_passesIntercepted20,rolling_sum_passingTDs20,rolling_sum_puntReturns20,rolling_sum_qbHurries20,rolling_sum_rushingAttempts20,rolling_sum_rushingTDs20,rolling_sum_rushingYards20,rolling_sum_sacks20,rolling_sum_tacklesForLoss20,rolling_sum_totalFumbles20,rolling_sum_totalPenaltiesYards20,rolling_sum_totalYards20,rolling_sum_turnovers20,rolling_sum_yardsPerPass20,rolling_sum_yardsPerRushAttempt20,rolling_sum_totalTDs20,rolling_sum_Points8,rolling_sum_firstDowns8,rolling_sum_fumblesLost8,rolling_sum_fumblesRecovered8,rolling_sum_interceptions8,rolling_sum_kickReturnYards8,rolling_sum_kickingPoints8,rolling_sum_netPassingYards8,rolling_sum_passesDeflected8,rolling_sum_passesIntercepted8,rolling_sum_passingTDs8,rolling_sum_puntReturns8,rolling_sum_qbHurries8,rolling_sum_rushingAttempts8,rolling_sum_rushingTDs8,rolling_sum_rushingYards8,rolling_sum_sacks8,rolling_sum_tacklesForLoss8,rolling_sum_totalFumbles8,rolling_sum_totalPenaltiesYards8,rolling_sum_totalYards8,rolling_sum_turnovers8,rolling_sum_yardsPerPass8,rolling_sum_yardsPerRushAttempt8,rolling_sum_totalTDs8,rolling_sum_Points20_opp,rolling_sum_firstDowns20_opp,rolling_sum_fumblesLost20_opp,rolling_sum_fumblesRecovered20_opp,rolling_sum_interceptions20_opp,rolling_sum_kickReturnYards20_opp,rolling_sum_kickingPoints20_opp,rolling_sum_netPassingYards20_opp,rolling_sum_passesDeflected20_opp,rolling_sum_passesIntercepted20_opp,rolling_sum_passingTDs20_opp,rolling_sum_puntReturns20_opp,rolling_sum_qbHurries20_opp,rolling_sum_rushingAttempts20_opp,rolling_sum_rushingTDs20_opp,rolling_sum_rushingYards20_opp,rolling_sum_sacks20_opp,rolling_sum_tacklesForLoss20_opp,rolling_sum_totalFumbles20_opp,rolling_sum_totalPenaltiesYards20_opp,rolling_sum_totalYards20_opp,rolling_sum_turnovers20_opp,rolling_sum_yardsPerPass20_opp,rolling_sum_yardsPerRushAttempt20_opp,rolling_sum_totalTDs20_opp,rolling_sum_Points8_opp,rolling_sum_firstDowns8_opp,rolling_sum_fumblesLost8_opp,rolling_sum_fumblesRecovered8_opp,rolling_sum_interceptions8_opp,rolling_sum_kickReturnYards8_opp,rolling_sum_kickingPoints8_opp,rolling_sum_netPassingYards8_opp,rolling_sum_passesDeflected8_opp,rolling_sum_passesIntercepted8_opp,rolling_sum_passingTDs8_opp,rolling_sum_puntReturns8_opp,rolling_sum_qbHurries8_opp,rolling_sum_rushingAttempts8_opp,rolling_sum_rushingTDs8_opp,rolling_sum_rushingYards8_opp,rolling_sum_sacks8_opp,rolling_sum_tacklesForLoss8_opp,rolling_sum_totalFumbles8_opp,rolling_sum_totalPenaltiesYar

In [100]:
for team, teamDF in teamDict.items():
    teamDict[team] = teamDF[teamDF.columns.drop(list(teamDF.filter(regex='Unnamed')))]

In [98]:
files = glob.glob(os.path.join('/users/blaizelahman/Desktop/CFB Model/Updated Data', '*'))

for file in files:
        
    try:
            
        os.remove(file)
        print(f"Deleted {file}")
            
    except Exception as e:
        
        print(f"Could not delete {file}: {e}")

Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Western_Michigan_updated_model 11.34.28 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Memphis_updated_model 11.34.24 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Arkansas_updated_model 11.34.22 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Wake_Forest_updated_model 11.34.28 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Texas_State_updated_model 11.34.27 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Liberty_updated_model 11.34.24 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Temple_updated_model 11.34.27 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Auburn_updated_model 11.34.22 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Miami_updated_model 11.34.24 PM.csv
Deleted /users/blaizelahman/Desktop/CFB Model/Updated Data/Alabama_updated_model 11.34.22 PM.csv
Deleted /users/b

In [102]:
for key, team in teamDict.items():
    if 2023 in team['Year'].values:
        name = key.replace(' ', '_') + '_updated_model.csv'
        path = os.path.join('/users/blaizelahman/Desktop/CFB Model/Updated Data', name)
        team.to_csv(path)
        print('CSV: ' + name)

CSV: Western_Michigan_updated_model.csv
CSV: Memphis_updated_model.csv
CSV: Arkansas_updated_model.csv
CSV: Wake_Forest_updated_model.csv
CSV: Texas_State_updated_model.csv
CSV: Temple_updated_model.csv
CSV: Auburn_updated_model.csv
CSV: Miami_updated_model.csv
CSV: Alabama_updated_model.csv
CSV: Louisville_updated_model.csv
CSV: Washington_State_updated_model.csv
CSV: North_Texas_updated_model.csv
CSV: Washington_updated_model.csv
CSV: Virginia_Tech_updated_model.csv
CSV: Texas_A&M_updated_model.csv
CSV: Buffalo_updated_model.csv
CSV: Ohio_updated_model.csv
CSV: Minnesota_updated_model.csv
CSV: Syracuse_updated_model.csv
CSV: Southern_Mississippi_updated_model.csv
CSV: Ole_Miss_updated_model.csv
CSV: Idaho_updated_model.csv
CSV: Western_Kentucky_updated_model.csv
CSV: Utah_updated_model.csv
CSV: Stanford_updated_model.csv
CSV: Georgia_State_updated_model.csv
CSV: Utah_State_updated_model.csv
CSV: West_Virginia_updated_model.csv
CSV: Nevada_updated_model.csv
CSV: Louisiana_updated_mode